# Subject-Driven Image Generation with BOFT (DreamBooth)

## Mini-Project: Parameter-Efficient Finetuning for Pretrained Models

This notebook demonstrates using **BOFT** (Butterfly Orthogonal Fine-Tuning) from the [PEFT](https://huggingface.co/docs/peft) library to finetune a pretrained **Stable Diffusion** model for subject-driven image generation via **DreamBooth**.

### Method Overview

**BOFT** inserts trainable full-rank orthogonal matrices with butterfly factorization structure into attention layers of the UNet. Only these small adapter parameters are trained while all pretrained weights stay frozen.

**Key properties:**
- **Parameter-efficient**: Only ~0.8% of total parameters are trainable
- **Orthogonal**: Preserves the hyperspherical energy of pretrained features
- **Butterfly structure**: Enables full-rank updates with fewer parameters than vanilla OFT

**References:**
- [BOFT Paper (ICLR 2024)](https://arxiv.org/abs/2311.06243) | [OFT Paper](https://arxiv.org/abs/2306.07280) | [PEFT Library](https://huggingface.co/docs/peft)

In [ ]:
# Install dependencies (uncomment if running for the first time)
# !pip install torch torchvision transformers accelerate diffusers peft Pillow matplotlib tqdm safetensors psutil

import os
import gc
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from diffusers import (
    AutoencoderKL,
    DDIMScheduler,
    DPMSolverMultistepScheduler,
    StableDiffusionPipeline,
    UNet2DConditionModel,
)
from transformers import CLIPTextModel, AutoTokenizer
from peft import BOFTConfig, get_peft_model, PeftModel
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# HuggingFace Login (required for downloading models)
# 1. Create a free account at https://huggingface.co/join
# 2. Generate a token at https://huggingface.co/settings/tokens (select "Read" access)
# 3. Paste your token below:

from huggingface_hub import login
login(token='YOUR_HF_TOKEN_HERE')  # <-- Replace with your actual token

## 1. Configuration

Set the model, BOFT hyperparameters, and data paths. We use the **dog** subject from the [Google DreamBooth dataset](https://github.com/google/dreambooth).

In [ ]:
# For mainland China: uncomment the mirror line below AND login first (see next cell)
# os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# ==================== Model ====================
MODEL_NAME = 'stabilityai/stable-diffusion-2-1'
UNIQUE_TOKEN = 'sks'
CLASS_TOKEN = 'dog'
INSTANCE_PROMPT = f'a photo of {UNIQUE_TOKEN} {CLASS_TOKEN}'
CLASS_PROMPT = f'a photo of {CLASS_TOKEN}'

# ==================== Paths ====================
DATA_DIR = './data'
INSTANCE_DIR = os.path.join(DATA_DIR, 'dreambooth', 'dataset', 'dog')
CLASS_DIR = os.path.join(DATA_DIR, 'class_data', CLASS_TOKEN)
OUTPUT_DIR = os.path.join(DATA_DIR, 'output', 'boft')

# ==================== Training ====================
RESOLUTION = 512
MAX_TRAIN_STEPS = 800
LEARNING_RATE = 3e-5
TRAIN_BATCH_SIZE = 1
NUM_CLASS_IMAGES = 100
CHECKPOINT_STEPS = 200
PRIOR_LOSS_WEIGHT = 1.0

# ==================== BOFT ====================
BOFT_BLOCK_NUM = 8
BOFT_BLOCK_SIZE = 0
BOFT_N_BUTTERFLY_FACTOR = 1
BOFT_DROPOUT = 0.1
UNET_TARGET_MODULES = ['to_q', 'to_v', 'to_k', 'to_out.0']

# ==================== Device ====================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WEIGHT_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print(f'Device: {DEVICE}')
print(f'Weight dtype: {WEIGHT_DTYPE}')
print(f'Instance prompt: "{INSTANCE_PROMPT}"')
print(f'Class prompt: "{CLASS_PROMPT}"')

## 2. Data Preparation

Download the DreamBooth dataset and display the training images for the selected subject.

In [ ]:
import shutil
from datasets import load_dataset

os.makedirs(INSTANCE_DIR, exist_ok=True)

existing = [p for p in Path(INSTANCE_DIR).iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
if len(existing) == 0:
    print('Downloading dog images from HuggingFace (diffusers/dog-example)...')
    try:
        ds = load_dataset('diffusers/dog-example', split='train')
        for i, example in enumerate(ds):
            example['image'].save(os.path.join(INSTANCE_DIR, f'dog_{i:02d}.jpg'))
        print(f'Downloaded {len(ds)} images.')
    except Exception as e:
        print(f'HuggingFace download failed: {e}')
        print('Please manually place your dog images into:', INSTANCE_DIR)
else:
    print(f'Found {len(existing)} instance images, skipping download.')

instance_images = sorted([p for p in Path(INSTANCE_DIR).iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')])
print(f'Number of training images: {len(instance_images)}')

fig, axes = plt.subplots(1, min(len(instance_images), 6), figsize=(18, 3))
if not isinstance(axes, np.ndarray):
    axes = [axes]
for ax, img_path in zip(axes, instance_images[:6]):
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')
fig.suptitle(f'Training Images ({UNIQUE_TOKEN} {CLASS_TOKEN})', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Baseline Generation (Before Finetuning)

Generate images with the **pretrained** Stable Diffusion model (before any finetuning) to establish a baseline for comparison.

In [ ]:
baseline_prompts = [
    f'a photo of {UNIQUE_TOKEN} {CLASS_TOKEN}',
    f'a {UNIQUE_TOKEN} {CLASS_TOKEN} in the snow',
    f'a {UNIQUE_TOKEN} {CLASS_TOKEN} on the beach',
    f'a {UNIQUE_TOKEN} {CLASS_TOKEN} wearing a red hat',
]

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_NAME, torch_dtype=WEIGHT_DTYPE, safety_checker=None
).to(DEVICE)
pipe.set_progress_bar_config(disable=False)

generator = torch.Generator(device=DEVICE).manual_seed(42)
baseline_images = []
for prompt in tqdm(baseline_prompts, desc='Generating baselines'):
    image = pipe(prompt, num_inference_steps=50, guidance_scale=7.5, generator=generator).images[0]
    baseline_images.append(image)

fig, axes = plt.subplots(1, len(baseline_images), figsize=(20, 5))
for ax, img, prompt in zip(axes, baseline_images, baseline_prompts):
    ax.imshow(img)
    ax.set_title(prompt, fontsize=9)
    ax.axis('off')
fig.suptitle('Baseline: Before BOFT Finetuning', fontsize=14)
plt.tight_layout()
plt.savefig('baseline_images.png', dpi=150, bbox_inches='tight')
plt.show()

del pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Baseline pipeline released.')

## 4. BOFT Finetuning Setup

Load model components, apply BOFT adapters to the UNet, and prepare the training pipeline.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, subfolder='tokenizer', use_fast=False)
noise_scheduler = DDIMScheduler.from_pretrained(MODEL_NAME, subfolder='scheduler')
text_encoder = CLIPTextModel.from_pretrained(MODEL_NAME, subfolder='text_encoder')
vae = AutoencoderKL.from_pretrained(MODEL_NAME, subfolder='vae')
unet = UNet2DConditionModel.from_pretrained(MODEL_NAME, subfolder='unet')

boft_config = BOFTConfig(
    boft_block_num=BOFT_BLOCK_NUM,
    boft_block_size=BOFT_BLOCK_SIZE,
    boft_n_butterfly_factor=BOFT_N_BUTTERFLY_FACTOR,
    target_modules=UNET_TARGET_MODULES,
    boft_dropout=BOFT_DROPOUT,
    bias='boft_only',
)
unet = get_peft_model(unet, boft_config)
unet.print_trainable_parameters()

vae.requires_grad_(False)
text_encoder.requires_grad_(False)

vae.to(DEVICE, dtype=WEIGHT_DTYPE)
text_encoder.to(DEVICE, dtype=WEIGHT_DTYPE)
unet.to(DEVICE, dtype=WEIGHT_DTYPE)
print('Models loaded and BOFT applied.')

### Generate Class Images for Prior Preservation

Prior preservation prevents the model from forgetting the general concept of the class (e.g., "dog") while learning the specific subject. We generate class images using the pretrained model.

In [ ]:
os.makedirs(CLASS_DIR, exist_ok=True)
existing_class_images = len(list(Path(CLASS_DIR).iterdir())) if os.path.exists(CLASS_DIR) else 0

if existing_class_images < NUM_CLASS_IMAGES:
    num_to_generate = NUM_CLASS_IMAGES - existing_class_images
    print(f'Generating {num_to_generate} class images with prompt: "{CLASS_PROMPT}"...')

    class_pipe = StableDiffusionPipeline.from_pretrained(
        MODEL_NAME, torch_dtype=WEIGHT_DTYPE, safety_checker=None
    ).to(DEVICE)
    class_pipe.set_progress_bar_config(disable=True)

    for i in tqdm(range(existing_class_images, NUM_CLASS_IMAGES), desc='Generating class images'):
        image = class_pipe(CLASS_PROMPT, num_inference_steps=30).images[0]
        image.save(os.path.join(CLASS_DIR, f'class_{i:04d}.jpg'))

    del class_pipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('Class images generated.')
else:
    print(f'Found {existing_class_images} existing class images, skipping generation.')

### Create Dataset and Optimizer

In [ ]:
class DreamBoothDataset(Dataset):
    def __init__(self, instance_data_root, instance_prompt, tokenizer,
                 class_data_root=None, class_prompt=None, size=512):
        self.tokenizer = tokenizer
        self.size = size
        self.instance_images_path = sorted([
            p for p in Path(instance_data_root).iterdir()
            if p.suffix.lower() in ('.jpg', '.jpeg', '.png')
        ])
        self.num_instance_images = len(self.instance_images_path)
        self.instance_prompt = instance_prompt
        self._length = self.num_instance_images

        if class_data_root is not None:
            self.class_images_path = sorted(list(Path(class_data_root).iterdir()))
            self.num_class_images = len(self.class_images_path)
            self._length = max(self.num_class_images, self.num_instance_images)
            self.class_prompt = class_prompt
        else:
            self.class_images_path = None

        self.image_transforms = transforms.Compose([
            transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return self._length

    def __getitem__(self, index):
        example = {}
        img = Image.open(self.instance_images_path[index % self.num_instance_images]).convert('RGB')
        example['instance_images'] = self.image_transforms(img)
        example['instance_prompt_ids'] = self.tokenizer(
            self.instance_prompt, truncation=True, padding='max_length',
            max_length=self.tokenizer.model_max_length, return_tensors='pt'
        ).input_ids
        if self.class_images_path:
            cls_img = Image.open(self.class_images_path[index % self.num_class_images]).convert('RGB')
            example['class_images'] = self.image_transforms(cls_img)
            example['class_prompt_ids'] = self.tokenizer(
                self.class_prompt, truncation=True, padding='max_length',
                max_length=self.tokenizer.model_max_length, return_tensors='pt'
            ).input_ids
        return example

def collate_fn(examples):
    input_ids = [e['instance_prompt_ids'] for e in examples]
    pixel_values = [e['instance_images'] for e in examples]
    input_ids += [e['class_prompt_ids'] for e in examples]
    pixel_values += [e['class_images'] for e in examples]
    pixel_values = torch.stack(pixel_values).to(memory_format=torch.contiguous_format).float()
    input_ids = torch.cat(input_ids, dim=0)
    return {'input_ids': input_ids, 'pixel_values': pixel_values}

train_dataset = DreamBoothDataset(
    instance_data_root=INSTANCE_DIR,
    instance_prompt=INSTANCE_PROMPT,
    class_data_root=CLASS_DIR,
    class_prompt=CLASS_PROMPT,
    tokenizer=tokenizer,
    size=RESOLUTION,
)
train_dataloader = DataLoader(
    train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=0,
)

params_to_optimize = [p for p in unet.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params_to_optimize, lr=LEARNING_RATE, weight_decay=1e-2)

total_params = sum(p.numel() for p in unet.parameters())
trainable_params = sum(p.numel() for p in params_to_optimize)
print(f'Dataset size: {len(train_dataset)}')
print(f'Total UNet params: {total_params:,}')
print(f'Trainable BOFT params: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)')

## 5. Training

Run the BOFT finetuning loop with prior preservation loss. Checkpoints are saved every 200 steps.

In [ ]:
unet.train()
losses = []
progress_bar = tqdm(range(MAX_TRAIN_STEPS), desc='Training')
train_iter = iter(train_dataloader)

for step in progress_bar:
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_dataloader)
        batch = next(train_iter)

    latents = vae.encode(
        batch['pixel_values'].to(DEVICE, dtype=WEIGHT_DTYPE)
    ).latent_dist.sample()
    latents = latents * vae.config.scaling_factor

    noise = torch.randn_like(latents)
    bsz = latents.shape[0]
    timesteps = torch.randint(
        0, noise_scheduler.config.num_train_timesteps, (bsz,), device=DEVICE
    ).long()
    noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

    encoder_hidden_states = text_encoder(batch['input_ids'].to(DEVICE))[0]
    model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

    if noise_scheduler.config.prediction_type == 'epsilon':
        target = noise
    elif noise_scheduler.config.prediction_type == 'v_prediction':
        target = noise_scheduler.get_velocity(latents, noise, timesteps)

    model_pred_inst, model_pred_prior = torch.chunk(model_pred, 2, dim=0)
    target_inst, target_prior = torch.chunk(target, 2, dim=0)
    instance_loss = F.mse_loss(model_pred_inst.float(), target_inst.float())
    prior_loss = F.mse_loss(model_pred_prior.float(), target_prior.float())
    loss = instance_loss + PRIOR_LOSS_WEIGHT * prior_loss

    loss.backward()
    torch.nn.utils.clip_grad_norm_(params_to_optimize, 1.0)
    optimizer.step()
    optimizer.zero_grad()

    losses.append(loss.detach().item())
    progress_bar.set_postfix(loss=f'{loss.item():.4f}')

    if (step + 1) % CHECKPOINT_STEPS == 0:
        ckpt_dir = os.path.join(OUTPUT_DIR, f'unet/{step + 1}')
        os.makedirs(ckpt_dir, exist_ok=True)
        unet.save_pretrained(os.path.join(OUTPUT_DIR, f'unet/{step + 1}'))
        print(f'\n  Checkpoint saved at step {step + 1}')

print(f'\nTraining complete! Final loss: {losses[-1]:.4f}')

## 6. Training Loss Visualization

Plot the training loss curve to verify convergence.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.plot(losses, alpha=0.3, color='steelblue', linewidth=0.5)
ax1.set_xlabel('Training Step')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss (Raw)')
ax1.grid(True, alpha=0.3)

window = min(50, len(losses) // 4)
if window > 1:
    smoothed = np.convolve(losses, np.ones(window) / window, mode='valid')
    ax2.plot(range(window - 1, len(losses)), smoothed, color='crimson', linewidth=1.5)
else:
    ax2.plot(losses, color='crimson', linewidth=1.5)
ax2.set_xlabel('Training Step')
ax2.set_ylabel('Loss (Smoothed)')
ax2.set_title(f'Training Loss (Moving Avg, window={window})')
ax2.grid(True, alpha=0.3)

plt.suptitle('BOFT DreamBooth Training Loss', fontsize=14)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Initial loss: {losses[0]:.4f}')
print(f'Final loss (last 10 avg): {np.mean(losses[-10:]):.4f}')

## 7. Inference After Finetuning

Generate images using the finetuned model with the same prompts used for the baseline.

In [ ]:
unet.eval()

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_NAME, torch_dtype=WEIGHT_DTYPE, safety_checker=None
).to(DEVICE)
pipe.unet = unet
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

finetuned_prompts = baseline_prompts.copy()
generator = torch.Generator(device=DEVICE).manual_seed(42)
finetuned_images = []

for prompt in tqdm(finetuned_prompts, desc='Generating finetuned images'):
    image = pipe(
        prompt, num_inference_steps=50, guidance_scale=7.5,
        generator=generator, negative_prompt='low quality, blurry, unfinished'
    ).images[0]
    finetuned_images.append(image)

fig, axes = plt.subplots(1, len(finetuned_images), figsize=(20, 5))
for ax, img, prompt in zip(axes, finetuned_images, finetuned_prompts):
    ax.imshow(img)
    ax.set_title(prompt, fontsize=9)
    ax.axis('off')
fig.suptitle('After BOFT Finetuning', fontsize=14)
plt.tight_layout()
plt.savefig('finetuned_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Qualitative Comparison: Before vs. After BOFT

Side-by-side comparison of generated images before and after finetuning with BOFT. The finetuned model should generate images that closely resemble the training subject while following the text prompts.

In [ ]:
n = len(baseline_prompts)
fig, axes = plt.subplots(2, n, figsize=(5 * n, 10))

for i in range(n):
    axes[0, i].imshow(baseline_images[i])
    axes[0, i].set_title(baseline_prompts[i], fontsize=9)
    axes[0, i].axis('off')

    axes[1, i].imshow(finetuned_images[i])
    axes[1, i].set_title(finetuned_prompts[i], fontsize=9)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Before BOFT', fontsize=14, labelpad=10)
axes[1, 0].set_ylabel('After BOFT', fontsize=14, labelpad=10)
fig.suptitle('Before vs. After BOFT Finetuning', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('comparison_before_after.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Extended Generation with Various Prompts

Generate images with diverse prompts to demonstrate the finetuned model's ability to place the learned subject in different contexts, styles, and scenarios.

In [ ]:
extra_prompts = [
    f'a {UNIQUE_TOKEN} {CLASS_TOKEN} in the jungle',
    f'a {UNIQUE_TOKEN} {CLASS_TOKEN} with a city in the background',
    f'a {UNIQUE_TOKEN} {CLASS_TOKEN} on a cobblestone street',
    f'a {UNIQUE_TOKEN} {CLASS_TOKEN} wearing a santa hat',
    f'a {UNIQUE_TOKEN} {CLASS_TOKEN} in a firefighter outfit',
    f'a {UNIQUE_TOKEN} {CLASS_TOKEN} wearing pink glasses',
    f'a red {UNIQUE_TOKEN} {CLASS_TOKEN}',
    f'a {UNIQUE_TOKEN} {CLASS_TOKEN} floating on top of water',
]

generator = torch.Generator(device=DEVICE).manual_seed(123)
extra_images = []
for prompt in tqdm(extra_prompts, desc='Generating extra images'):
    image = pipe(
        prompt, num_inference_steps=50, guidance_scale=7.5,
        generator=generator, negative_prompt='low quality, blurry, unfinished'
    ).images[0]
    extra_images.append(image)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for idx, (ax, img, prompt) in enumerate(zip(axes.flat, extra_images, extra_prompts)):
    ax.imshow(img)
    ax.set_title(prompt, fontsize=9)
    ax.axis('off')
fig.suptitle('Generated Images with Various Prompts (After BOFT Finetuning)', fontsize=14)
plt.tight_layout()
plt.savefig('multi_prompt_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary

### BOFT Configuration
| Parameter | Value |
|-----------|-------|
| Block Num | 8 |
| Butterfly Factor | 1 |
| Dropout | 0.1 |
| Target Modules | to_q, to_v, to_k, to_out.0 |
| Bias | boft_only |

### Training Configuration
| Parameter | Value |
|-----------|-------|
| Base Model | stabilityai/stable-diffusion-2-1 |
| Subject | dog (DreamBooth dataset) |
| Max Steps | 800 |
| Learning Rate | 3e-5 |
| Resolution | 512 |
| Prior Preservation | Yes (weight=1.0) |

### Key Observations
- BOFT enables subject-driven generation with minimal trainable parameters (~0.8%)
- The orthogonal constraint preserves the pretrained model's generalization ability
- The finetuned model successfully learns the specific subject identity while maintaining prompt-following capability
- Prior preservation loss helps prevent catastrophic forgetting of the general class concept